!pip install -q \
    torch \
    lightning \
    omegaconf \
    tokenizers \
    datasets \
    tensorboard \
    clearml \
    pandas \
    numpy \
    tqdm

In [1]:
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()

dirs = [
    "configs",
    "data",
    "data/processed",
    "notebooks",
    "src",
    "src/data",
    "src/models",
    "src/training",
    "src/tokenization",
    "checkpoints",
    "logs",
    "tokenizers",
]

for d in dirs:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)

C:\Users\521\CPM Owl\2026\Modern_neural_net_arch\HW2


In [2]:
import os
import json
from pathlib import Path

import torch
import lightning.pytorch as pl
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from omegaconf import OmegaConf
from tokenizers import Tokenizer

from src.utils import set_seed, count_parameters
from src.data.packed_datamodule import PackedTextDataModule
from src.training.lit_gpt import LitGPTLanguageModel

##### Загрузка конфига

In [3]:
config_path = "configs/gpt_small.yaml"
cfg = OmegaConf.load(config_path)
cfg

{'project': {'name': 'lab2-gpt-like', 'experiment_name': 'gpt-small-packed-wikitext'}, 'paths': {'tokenizer_path': 'tokenizers/commoncrawl_bpe.json', 'text_file': 'data/processed/wikitext_lab2.jsonl', 'checkpoint_dir': 'checkpoints', 'log_dir': 'logs'}, 'data': {'block_size': 512, 'batch_size': 16, 'val_ratio': 0.05, 'num_workers': 0, 'max_texts': 20000, 'append_eos': True}, 'model': {'vocab_size': None, 'max_position_embeddings': 512, 'hidden_size': 384, 'num_hidden_layers': 6, 'num_attention_heads': 6, 'intermediate_size': 1536, 'dropout': 0.1, 'layer_norm_eps': 1e-05, 'tie_word_embeddings': True}, 'training': {'seed': 42, 'max_epochs': 10, 'learning_rate': 0.0003, 'weight_decay': 0.01, 'betas': [0.9, 0.95], 'warmup_steps': 1000, 'min_lr_ratio': 0.1, 'gradient_clip_val': 1.0, 'accumulate_grad_batches': 1, 'precision': '32', 'log_every_n_steps': 20, 'save_top_k': 1, 'monitor': 'val/perplexity', 'mode': 'min'}, 'generation': {'prompt': 'The history of natural language processing', 'max

##### Проверка токенизатора из ЛР1

In [4]:
tokenizer_path = Path(cfg.paths.tokenizer_path)

if not tokenizer_path.exists():
    raise FileNotFoundError(
        f"Не найден токенизатор {tokenizer_path}. "
        "Сначала выполните ЛР1 и сохраните BPE-токенизатор."
    )

tokenizer = Tokenizer.from_file(str(tokenizer_path))

vocab_size = tokenizer.get_vocab_size()
pad_id = tokenizer.token_to_id("<PAD>")
eos_id = tokenizer.token_to_id("<EOS>")

print("Vocab size:", vocab_size)
print("PAD id:", pad_id)
print("EOS id:", eos_id)

Vocab size: 8000
PAD id: 0
EOS id: 3


##### Подстановка размера словаря в конфиг

In [5]:
cfg.model.vocab_size = vocab_size

print(OmegaConf.to_yaml(cfg))

project:
  name: lab2-gpt-like
  experiment_name: gpt-small-packed-wikitext
paths:
  tokenizer_path: tokenizers/commoncrawl_bpe.json
  text_file: data/processed/wikitext_lab2.jsonl
  checkpoint_dir: checkpoints
  log_dir: logs
data:
  block_size: 512
  batch_size: 16
  val_ratio: 0.05
  num_workers: 0
  max_texts: 20000
  append_eos: true
model:
  vocab_size: 8000
  max_position_embeddings: 512
  hidden_size: 384
  num_hidden_layers: 6
  num_attention_heads: 6
  intermediate_size: 1536
  dropout: 0.1
  layer_norm_eps: 1.0e-05
  tie_word_embeddings: true
training:
  seed: 42
  max_epochs: 10
  learning_rate: 0.0003
  weight_decay: 0.01
  betas:
  - 0.9
  - 0.95
  warmup_steps: 1000
  min_lr_ratio: 0.1
  gradient_clip_val: 1.0
  accumulate_grad_batches: 1
  precision: '32'
  log_every_n_steps: 20
  save_top_k: 1
  monitor: val/perplexity
  mode: min
generation:
  prompt: The history of natural language processing
  max_new_tokens: 80
  temperature: 0.9
  top_k: 50



In [6]:
USE_CLEARML = True

clearml_task = None

if USE_CLEARML:
    from clearml import Task
    
    clearml_task = Task.init(
        project_name=cfg.project.name,
        task_name=cfg.project.experiment_name,
    )
    
    clearml_task.connect(OmegaConf.to_container(cfg, resolve=True))
    
    print("ClearML task initialized")
else:
    print("ClearML отключён")

MissingConfigError: It seems ClearML is not configured on this machine!
To get started with ClearML, setup your own 'clearml-server' or create a free account at https://app.clear.ml
Setup instructions can be found here: https://clear.ml/docs

##### Создание DataModule

In [7]:
set_seed(cfg.training.seed)

data_module = PackedTextDataModule(
    tokenizer_path=cfg.paths.tokenizer_path,
    text_file=cfg.paths.text_file,
    block_size=cfg.data.block_size,
    batch_size=cfg.data.batch_size,
    val_ratio=cfg.data.val_ratio,
    num_workers=cfg.data.num_workers,
    max_texts=cfg.data.max_texts,
    append_eos=cfg.data.append_eos,
    seed=cfg.training.seed,
)

data_module.prepare_data()
data_module.setup()

print("Train batches:", len(data_module.train_dataloader()))
print("Val batches:", len(data_module.val_dataloader()))

Train batches: 774
Val batches: 41


In [8]:
# Проверим батч
batch = next(iter(data_module.train_dataloader()))

for k, v in batch.items():
    print(k, v.shape, v.dtype)

print("input_ids example:")
print(batch["input_ids"][0][:80])

print("packed_mask example:")
print(batch["packed_mask"][0][:80])

input_ids torch.Size([16, 512]) torch.int64
packed_mask torch.Size([16, 512]) torch.int64
input_ids example:
tensor([   1,   17,   30,    1,    1,    1,    1,   30,    1,   30,    1,  183,
         909,  179,   30,    1,   22,   30,    1,   78,  785,   30,    1,   40,
        3011,   50, 6460, 3277,   30,    1,    1,    1, 1221, 2754,  545,   30,
           1,  850,  374,   30,    1, 4373, 1345,   47, 1386, 1257,   89,    9,
        1386,   30,    1,    1,    1,    1,   30,    1,    1,    1,   30,    1,
           1,   30,    1, 4270,  497, 1611,   89,    9,   30,    1,  183,  909,
         179,   30,    1,   30,    1,   23, 1174,  571])
packed_mask example:
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1])


##### Создание модели

In [9]:
model = LitGPTLanguageModel(
    OmegaConf.to_container(cfg, resolve=True)
)

params = count_parameters(model)

print(params)

{'total': 13719552, 'trainable': 13719552}


##### Проверка forward pass

In [10]:
model.eval()

with torch.no_grad():
    logits = model(
        input_ids=batch["input_ids"],
        packed_mask=batch["packed_mask"],
    )

print("Logits shape:", logits.shape)
# [batch_size, block_size, vocab_size]

Logits shape: torch.Size([16, 512, 8000])


##### Проверка одного training step

In [11]:
model.train()

loss = model.training_step(batch, batch_idx=0)

print("Loss:", loss.item())
print("Perplexity:", torch.exp(loss).item())

Loss: 9.00330638885498
Perplexity: 8129.92041015625


C:\Users\521\AppData\Roaming\Python\Python311\site-packages\lightning\pytorch\core\module.py:451: You are trying to `self.log()` but the `self.trainer` reference is not registered on the model yet. This is most likely because the model hasn't been passed to the `Trainer`


##### Callback для чекпоинтов

In [12]:
checkpoint_callback = ModelCheckpoint(
    dirpath=cfg.paths.checkpoint_dir,
    filename="gpt-like-{epoch:02d}-{val_perplexity:.2f}",
    monitor=cfg.training.monitor,
    mode=cfg.training.mode,
    save_top_k=cfg.training.save_top_k,
    save_last=True,
)

lr_monitor = LearningRateMonitor(
    logging_interval="step",
)

##### TensorBoard logger

In [13]:
tb_logger = TensorBoardLogger(
    save_dir=cfg.paths.log_dir,
    name=cfg.project.experiment_name,
)

##### Trainer

In [14]:
accelerator = "gpu" if torch.cuda.is_available() else "cpu"

trainer = pl.Trainer(
    max_epochs=cfg.training.max_epochs,
    accelerator=accelerator,
    devices=1,
    logger=tb_logger,
    callbacks=[
        checkpoint_callback,
        lr_monitor,
    ],
    gradient_clip_val=cfg.training.gradient_clip_val,
    accumulate_grad_batches=cfg.training.accumulate_grad_batches,
    precision=cfg.training.precision,
    log_every_n_steps=cfg.training.log_every_n_steps,
)

accelerator

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


'gpu'

##### Обучение модели

In [15]:
trainer.fit(
    model=model,
    datamodule=data_module,
)

You are using a CUDA device ('NVIDIA GeForce RTX 5070 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
C:\Users\521\AppData\Roaming\Python\Python311\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:881: Checkpoint directory C:\Users\521\CPM Owl\2026\Modern_neural_net_arch\HW2\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
C:\Users\521\AppData\Roaming\Python\Python311\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to

┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ GPTLikeModel │ 13.7 M │ train │     0 │
└───┴───────┴──────────────┴────────┴───────┴───────┘

Trainable params: 13.7 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 13.7 M                                                                                               
Total estimated model params size (MB): 54                                                                         
Modules in train mode: 91                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

C:\Users\521\AppData\Roaming\Python\Python311\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:
434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of 
the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=10` reached.


##### Лучший чекпоинт

In [16]:
best_ckpt_path = checkpoint_callback.best_model_path

print("Best checkpoint:")
print(best_ckpt_path)

Best checkpoint:
C:\Users\521\CPM Owl\2026\Modern_neural_net_arch\HW2\checkpoints\gpt-like-epoch=09-val_perplexity=0.00.ckpt


In [17]:
print("Best score:")
print(checkpoint_callback.best_model_score)

Best score:
tensor(24.9629, device='cuda:0')


##### Валидация лучшей модели

In [18]:
best_model = LitGPTLanguageModel.load_from_checkpoint(
    best_ckpt_path,
    config=OmegaConf.to_container(cfg, resolve=True),
)

val_results = trainer.validate(
    model=best_model,
    datamodule=data_module,
    ckpt_path=best_ckpt_path,
)

val_results

Restoring states from the checkpoint path at C:\Users\521\CPM Owl\2026\Modern_neural_net_arch\HW2\checkpoints\gpt-like-epoch=09-val_perplexity=0.00.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at C:\Users\521\CPM Owl\2026\Modern_neural_net_arch\HW2\checkpoints\gpt-like-epoch=09-val_perplexity=0.00.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         val/loss          │     3.210341691970825     │
│      val/perplexity       │     24.96294593811035     │
└───────────────────────────┴───────────────────────────┘

[{'val/loss': 3.210341691970825, 'val/perplexity': 24.96294593811035}]

##### Восстановление обучения из чекпоинта

In [19]:
# trainer.fit(
#     model=model,
#     datamodule=data_module,
#     ckpt_path=best_ckpt_path,
# )

##### Генерация текста
Для инференса packed batching не нужен: вся последовательность считается одним объектом, поэтому packed_mask заполняется единицами.

In [20]:
def top_k_filtering(logits, top_k: int):
    if top_k is None or top_k <= 0:
        return logits

    values, indices = torch.topk(logits, top_k)

    filtered = torch.full_like(logits, float("-inf"))
    filtered.scatter_(dim=-1, index=indices, src=values)

    return filtered


@torch.no_grad()
def generate_text(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 100,
    temperature: float = 1.0,
    top_k: int = 50,
    device: str = "cpu",
):
    model.eval()
    model.to(device)

    ids = tokenizer.encode(prompt).ids

    if len(ids) == 0:
        raise ValueError("Пустой prompt после токенизации.")

    input_ids = torch.tensor(
        [ids],
        dtype=torch.long,
        device=device,
    )

    for _ in range(max_new_tokens):
        if input_ids.size(1) > model.model.config.max_position_embeddings:
            input_ids_context = input_ids[:, -model.model.config.max_position_embeddings:]
        else:
            input_ids_context = input_ids

        packed_mask = torch.ones_like(input_ids_context)

        logits = model(
            input_ids=input_ids_context,
            packed_mask=packed_mask,
        )

        next_token_logits = logits[:, -1, :]

        if temperature <= 0:
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        else:
            next_token_logits = next_token_logits / temperature
            next_token_logits = top_k_filtering(next_token_logits, top_k=top_k)

            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_token], dim=1)

        eos_id = tokenizer.token_to_id("<EOS>")
        if eos_id is not None and next_token.item() == eos_id:
            break

    generated_ids = input_ids[0].tolist()
    text = tokenizer.decode(generated_ids)

    return text

##### Генерация из лучшего чекпоинта

In [21]:
device = "cuda" if torch.cuda.is_available() else "cpu"

best_model = LitGPTLanguageModel.load_from_checkpoint(
    best_ckpt_path,
    config=OmegaConf.to_container(cfg, resolve=True),
)

prompt = cfg.generation.prompt

generated = generate_text(
    model=best_model,
    tokenizer=tokenizer,
    prompt=prompt,
    max_new_tokens=cfg.generation.max_new_tokens,
    temperature=cfg.generation.temperature,
    top_k=cfg.generation.top_k,
    device=device,
)

print(generated)

he history of natural language processing was built in many at the hurch of other hristian  century  during the eat and the oad century 


##### Запуск TensorBoard

In [22]:
pip show tensorboard

Name: tensorboard
Version: 2.20.0
Summary: TensorBoard lets you watch Tensors Flow
Home-page: https://github.com/tensorflow/tensorboard
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: C:\Users\521\AppData\Roaming\Python\Python311\site-packages
Requires: absl-py, grpcio, markdown, numpy, packaging, pillow, protobuf, setuptools, tensorboard-data-server, werkzeug
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [23]:
os.environ['TENSORBOARD_BINARY'] = 'C:/Users/521/AppData/Roaming/Python/Python311/site-packages'

In [24]:
%reload_ext tensorboard

In [25]:
%tensorboard

ERROR: Failed to start
'C:/Users/521/AppData/Roaming/Python/Python311/site-packages' (set by
the `TENSORBOARD_BINARY` environment variable): [WinError 5] Отказано
в доступе

##### Результаты

In [26]:
summary = {
    "vocab_size": cfg.model.vocab_size,
    "block_size": cfg.data.block_size,
    "batch_size": cfg.data.batch_size,
    "hidden_size": cfg.model.hidden_size,
    "num_layers": cfg.model.num_hidden_layers,
    "num_heads": cfg.model.num_attention_heads,
    "intermediate_size": cfg.model.intermediate_size,
    "learning_rate": cfg.training.learning_rate,
    "warmup_steps": cfg.training.warmup_steps,
    "gradient_clip_val": cfg.training.gradient_clip_val,
    "best_checkpoint": best_ckpt_path,
    "best_val_perplexity": float(checkpoint_callback.best_model_score)
    if checkpoint_callback.best_model_score is not None
    else None,
}

summary

{'vocab_size': 8000,
 'block_size': 512,
 'batch_size': 16,
 'hidden_size': 384,
 'num_layers': 6,
 'num_heads': 6,
 'intermediate_size': 1536,
 'learning_rate': 0.0003,
 'warmup_steps': 1000,
 'gradient_clip_val': 1.0,
 'best_checkpoint': 'C:\\Users\\521\\CPM Owl\\2026\\Modern_neural_net_arch\\HW2\\checkpoints\\gpt-like-epoch=09-val_perplexity=0.00.ckpt',
 'best_val_perplexity': 24.96294593811035}